In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import pandas as pd
import numpy as np
import itertools
from collections import defaultdict

from unsampled_truth.metrics.anchors import anchor_distribution

from unsampled_truth.metrics.distribution_measures import average_pairwise_wasserstein_distance
import pandas as pd
import re

In [ ]:
# Analysis settings
QUESTIONNAIRE = "BFI"  # "BFI" or "SD3"
BASELINE_PERSONAS = ["antonia", "donovan", "kyle", "nicole", "renate"]

# Define the ordinal mapping
ordinal_mapping = {
    "Strongly Disagree": 1.0,
    "Disagree": 2.0,
    "Neutral": 3.0,
    "Agree": 4.0,
    "Strongly Agree": 5.0
}

def load_matrix_trials(filepath: str, baseline_persona: str) -> list:
    """
    Loads trials and extracts the prompt component levels of each Prompt Variant Matrix cell.
    """
    trials = []
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line)
            
            # Map option symbol probabilities onto semantic anchors
            scored_dist = anchor_distribution([data])
            
            # Skip empty distributions
            if sum(scored_dist.values()) == 0:
                continue

            # item_id has the form '<item>_<item wording>', e.g. 'BFI0_V1'
            item, item_wording = data["item_id"].split('_')
            task_instruction = data["task_instruction"]
            option_symbol = data["option_symbol"]
            persona_wording = data["persona_wording"]

            trials.append({
                "baseline_persona": baseline_persona,
                "persona_wording": persona_wording,
                "item": item,
                "item_wording": item_wording,
                "task_instruction": task_instruction,
                "option_symbol": option_symbol,
                "distribution": scored_dist
            })
            
    return trials

def compute_marginal_apwd(base_dir: str, model_name: str) -> pd.DataFrame:
    """
    Calculates the Marginal APWD for each prompt component using strict isolation.
    """
    base_path = Path(base_dir)
    all_records = []
    
    # 1. Load and flatten the Prompt Variant Matrices for the specified model
    for filepath in base_path.rglob("*_raw.jsonl"):
        parts = filepath.stem.split('_')
        if len(parts) < 2:
            continue
            
        baseline_persona = parts[0].capitalize()
        name = parts[1]

        if name.lower() != model_name.lower() or baseline_persona.lower() not in BASELINE_PERSONAS:
            continue
            
        all_records.extend(load_matrix_trials(filepath, baseline_persona))

    df = pd.DataFrame(all_records)
    
    if df.empty:
        print(f"No data found for model: {model_name}")
        return pd.DataFrame()
    df = df[df["item"].str.startswith(QUESTIONNAIRE)]

    results = []
    
    # 2. Define the isolation logic (Lock N-1, Vary 1)
    isolation_targets = {
        "Option Symbol": {
            "vary": "option_symbol",
            "lock": ["baseline_persona", "persona_wording", "item", "item_wording", "task_instruction"]
        },
        "Task Instruction": {
            "vary": "task_instruction",
            "lock": ["baseline_persona", "persona_wording", "item", "item_wording", "option_symbol"]
        },
        "Item Wording": {
            "vary": "item_wording",
            "lock": ["baseline_persona", "persona_wording", "item", "task_instruction", "option_symbol"]
        },
        "Persona Wording": {
            "vary": "persona_wording",
            "lock": ["baseline_persona", "item", "item_wording", "task_instruction", "option_symbol"]
        }
    }


    # 3. Compute Marginal APWD for each component
    for target_name, config in isolation_targets.items():
        locked_columns = config["lock"]
        
        # Group by the locked columns. The resulting group contains the variations of our target.
        grouped = df.groupby(locked_columns)
        
        marginal_apwds = []
        for _, group in grouped:
            distributions = group["distribution"].tolist()

            # Only compute if we have variations to compare (ideally all 5)
            if len(distributions) > 1:
                apwd = average_pairwise_wasserstein_distance(distributions, ordinal_mapping)
                marginal_apwds.append(apwd)
                
        # Average the APWDs across all locked states
        marginal_apwd_mean = np.mean(marginal_apwds) if marginal_apwds else 0.0
        # Calculate confidence intervals or standard deviation if needed for error bars in the future
        std_deviation = np.std(marginal_apwds) if marginal_apwds else 0.0
        
        results.append({
            "component": target_name,
            "marginal_apwd": marginal_apwd_mean,
            "marginal_apwd_std": std_deviation
        })

    # Convert to DataFrame and sort by Marginal APWD
    results_df = pd.DataFrame(results).sort_values(by="marginal_apwd", ascending=False)
    results_df.insert(0, "model", model_name)
    
    return results_df

In [ ]:
model_list = [
    "Llama-3.2-1B-Instruct", "Llama-3.2-3B-Instruct", "Llama-3.1-8B-Instruct", 
    "Qwen3-0.6B", "Qwen3-1.7B", "Qwen3-4B", "Qwen3-8B", "Qwen3-14B", 
    "gemma-4-E2B-it", "gemma-4-E4B-it",
    "Ministral-3-3B-Instruct-2512-BF16", "Ministral-3-8B-Instruct-2512-BF16", "Ministral-3-14B-Instruct-2512-BF16"
    ]
 
results_df = pd.DataFrame()
for model in model_list:
    results = compute_marginal_apwd("../../outputs/raw_results/", model)
    results_df = pd.concat([results_df, results], ignore_index=True)
results_df.to_csv(f"../../outputs/tables/marginal_apwd_{QUESTIONNAIRE.lower()}.csv", index=False)

In [ ]:
def compute_marginal_apwd_raw(base_dir: str, model_name: str) -> pd.DataFrame:
    """
    Calculates the Marginal APWD for each prompt component and returns the RAW distribution 
    (unaggregated) so it can be used for boxplots.
    """
    base_path = Path(base_dir)
    all_records = []
    
    # 1. Load and flatten the Prompt Variant Matrices for the specified model
    for filepath in base_path.rglob("*_raw.jsonl"):
        parts = filepath.stem.split('_')
        if len(parts) < 2:
            continue
            
        baseline_persona = parts[0].capitalize()
        name = parts[1]

        if name.lower() != model_name.lower() or baseline_persona.lower() not in BASELINE_PERSONAS:
            continue
            
        all_records.extend(load_matrix_trials(filepath, baseline_persona))

    df = pd.DataFrame(all_records)

    if df.empty:
        print(f"No data found for model: {model_name}")
        return pd.DataFrame()

    # keep only items of the selected questionnaire
    print(f"Initial records for {model_name}: {len(df)}")
    df = df[df["item"].str.startswith(QUESTIONNAIRE)]
    print(f"Records after filtering for {model_name}: {len(df)}")

    raw_results = []
    
    # 2. Define the isolation logic
    isolation_targets = {
        "Option Symbol": {
            "vary": "option_symbol",
            "lock": ["baseline_persona", "persona_wording", "item", "item_wording", "task_instruction"]
        },
        "Task Instruction": {
            "vary": "task_instruction",
            "lock": ["baseline_persona", "persona_wording", "item", "item_wording", "option_symbol"]
        },
        "Item Wording": {
            "vary": "item_wording",
            "lock": ["baseline_persona", "persona_wording", "item", "task_instruction", "option_symbol"]
        },
        "Persona Wording": {
            "vary": "persona_wording",
            "lock": ["baseline_persona", "item", "item_wording", "task_instruction", "option_symbol"]
        }
    }

    # 3. Compute Marginal APWD and log EVERY measurement
    for target_name, config in isolation_targets.items():
        locked_columns = config["lock"]
        grouped = df.groupby(locked_columns)
        
        for _, group in grouped:
            distributions = group["distribution"].tolist()

            if len(distributions) > 1:
                apwd = average_pairwise_wasserstein_distance(distributions, ordinal_mapping)
                # Append the raw measurement, not the mean
                raw_results.append({
                    "model": model_name,
                    "component": target_name,
                    "marginal_apwd": apwd
                })

    return pd.DataFrame(raw_results)

In [ ]:
# 1. Gather all raw data
results_df = pd.DataFrame()
for model in model_list:
    print(f"Processing {model}...")
    results = compute_marginal_apwd_raw("../../outputs/raw_results/", model)
    results_df = pd.concat([results_df, results], ignore_index=True)

In [ ]:

# 2. Extract Family and Size
def extract_model_info(model_name):
    model_name = model_name.lower()
    if "llama" in model_name: family = "Llama 3.1/3.2"
    elif "qwen" in model_name: family = "Qwen3"
    elif "gemma" in model_name: family = "Gemma-4"
    elif "ministral" in model_name: family = "Ministral-3"
    else: family = "Other"
    
    match = re.search(r'(\d+(?:\.\d+)?)b', model_name)
    size = float(match.group(1)) if match else 0.0
    return family, size

results_df[["model_family", "parameter_size"]] = results_df["model"].apply(
    lambda x: pd.Series(extract_model_info(x))
)

# 3. String Truncation: Cut everything after the 'B' or 'b'
results_df["model_short"] = results_df["model"].apply(lambda x: re.sub(r'([bB]).*', r'\1', x))

# 4. Component Ordering
# Define the exact categorical orders you requested
component_order = ["Persona Wording", "Task Instruction", "Item Wording", "Option Symbol"]
family_order = ["Llama 3.1/3.2", "Qwen3", "Ministral-3", "Gemma-4"]

# Sort the dataframe so the X-axis naturally orders by parameter size within families
results_df = results_df.sort_values(by=["model_family", "parameter_size"])

In [ ]:
# Clean up the X-axis: Remove the family name since the facet title already shows it
def clean_model_name(name):
    # Extracts just the parameter size (e.g., "1B", "0.6B", "14B")
    match = re.search(r'(\d+(?:\.\d+)?B)', name, re.IGNORECASE)
    return match.group(1).upper() if match else name

results_df["model_short"] = results_df["model_short"].apply(clean_model_name)

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.0,
})

In [ ]:
g = sns.catplot(
    data=results_df,
    kind="box",
    x="model_short",
    y="marginal_apwd",
    hue="component",
    col="model_family",
    col_wrap=2,               # Forces the 2x2 grid
    col_order=family_order, 
    hue_order=component_order,
    palette=["#26A69A", "#EF5350", "#5C6BC0", "#FFA726"], # Custom colors for better distinction
    height=2.5,               # Reduced height to keep overall figure compact
    aspect=1.5,               # Significantly widened to uncramp the boxes
    linewidth=1.0,
    fliersize=0,
    sharex=False,
    sharey=True               # Ensures Y-axis remains comparable across all facets
)

g.set_axis_labels("", "Marginal APWD")
g.set_titles(col_template="{col_name}", fontweight='bold')

# Anchor legend closely to the top of the 2x2 grid
sns.move_legend(
    g, "lower center",
    bbox_to_anchor=(0.5, 0.98), # Lowered from 1.05 to close the gap
    ncol=4,
    title=None,
    frameon=False
)

sns.despine(top=True, right=True)
# Tighten layout: hspace controls vertical gap between rows, top accommodates legend
g.fig.subplots_adjust(top=0.88, hspace=0.3, wspace=0.05) 

for ax in g.axes.flat:
    ax.yaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.5)
    ax.set_axisbelow(True)

plt.savefig(f"../../outputs/figures/marginal_apwd_boxplot_2x2_{QUESTIONNAIRE.lower()}.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 1. Generate the 1x4 Faceted Boxplot (Geometry matched to relplot)
g = sns.catplot(
    data=results_df,
    kind="box",
    x="model_short",
    y="marginal_apwd",
    hue="component",
    col="model_family",
    col_order=family_order, 
    hue_order=component_order,
    palette=["#26A69A", "#EF5350", "#5C6BC0", "#FFA726"],
    height=3.2,               # Synchronized with relplot
    aspect=0.9,               # Synchronized with relplot 
    linewidth=1.0,
    fliersize=0,
    sharex=False
)

# 2. Clean Titles and Labels
g.set_axis_labels("", "Marginal APWD")
g.set_titles(col_template="{col_name}")

# 3. Top-Anchored Legend (Synchronized with relplot)
sns.move_legend(
    g, "lower center",
    bbox_to_anchor=(0.45, 1.05), # Snug against the top, slightly left-offset
    ncol=4,
    title=None,
    frameon=False
)

sns.despine(top=True, right=True)

# 4. Spacing Adjustment (Synchronized with relplot)
g.fig.subplots_adjust(top=0.95, wspace=0.05) 

# 5. Grid Formatting and Label Rotation
for ax in g.axes.flat:
    ax.yaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.5)
    ax.set_axisbelow(True) 
    
    # Pragmatic fix: Rotate x-labels to prevent overlap in the narrower 0.9 aspect panels
    ax.tick_params(axis='x', rotation=45)
# rename legend labels to be more concise and clear
g._legend.set_title(None)
#g._legend.texts[0].set_text("Persona Variant")
#g._legend.texts[1].set_text("Instruction Variant")
#g._legend.texts[2].set_text("Item Variant")
#g._legend.texts[3].set_text("Option Symbol Variant")

plt.savefig(f"../../outputs/figures/marginal_apwd_boxplot_1x4_{QUESTIONNAIRE.lower()}.pdf", format="pdf", bbox_inches="tight")
plt.show()